In [1]:
import numpy as np
import pandas as pd
from matplotlib import style
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,MinMaxScaler
from numpy import array
from sklearn.metrics import accuracy_score
from numpy import argmax
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import confusion_matrix, classification_report, precision_score, recall_score, roc_curve, auc
from sklearn.preprocessing import StandardScaler
from sklearn import preprocessing
from keras.utils import np_utils
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score


import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation
from tensorflow.keras.layers import Embedding
from tensorflow.keras.layers import Conv1D, GlobalMaxPooling1D
from tensorflow.keras.datasets import imdb
import numpy as np
import pandas as pd
import os
import gc


#import libraries
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc
from sklearn import metrics
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation,Dropout
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import SGD,Adam
import keras
# Conv1D + LSTM
from keras.layers.convolutional import Conv1D,MaxPooling1D
from tensorflow.keras.layers import Embedding, Dense, Concatenate, Conv1D, Bidirectional, LSTM, GlobalAveragePooling1D, GlobalMaxPooling1D
from keras.layers import LSTM
from keras.layers import Dense,Dropout
from keras.layers import Flatten
from keras.layers import Input
from keras import Input, Model
from keras.layers import concatenate


from sklearn.preprocessing import LabelEncoder
from sklearn import preprocessing
from keras.utils import np_utils
import numpy as np
import pandas as pd
import random
from itertools import chain
from sklearn.model_selection import train_test_split
from sklearn import svm
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn import metrics
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import KFold, cross_val_score
from sklearn import preprocessing
label_encoder = preprocessing.LabelEncoder()


from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from math import sqrt


import numpy as np
import pandas as pd
import os
import gc
#import matplotlib.pyplot as plt


#import libraries
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc
from sklearn import metrics
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation,Dropout
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import SGD,Adam
import keras
# Conv1D + LSTM
from keras.layers.convolutional import Conv1D,MaxPooling1D,AveragePooling1D
from tensorflow.keras.layers import Embedding, Dense, Concatenate, Conv1D, Bidirectional, LSTM, GlobalAveragePooling1D, GlobalMaxPooling1D
from keras.layers import LSTM
from keras.layers import Dense,Dropout
from keras.layers import Flatten
from keras.layers import Input
from keras import Input, Model
from keras.layers import concatenate

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation,Dropout
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import SGD,Adam

import math

from keras import backend as K

#from keras import initializations
from keras import initializers, regularizers, constraints
from tensorflow.keras.layers import Layer

from tensorflow.keras import backend as K
from tensorflow.keras import initializers, regularizers, constraints
from tensorflow.keras.layers import Layer

pip install numpy==1.26.4

In [2]:
# Create an attention class

class attention(Layer):
    def __init__(self, step_dim,
                 W_regularizer=None, b_regularizer=None,
                 W_constraint=None, b_constraint=None,
                 bias=True, **kwargs):
        """
        Keras Layer that implements an Attention mechanism for temporal data.
        Supports Masking.
        Follows the work of Raffel et al. [https://arxiv.org/abs/1512.08756]
        # Input shape
            3D tensor with shape: `(samples, steps, features)`.
        # Output shape
            2D tensor with shape: `(samples, features)`.
        :param kwargs:
        Just put it on top of an RNN Layer (GRU/LSTM/SimpleRNN) with return_sequences=True.
        The dimensions are inferred based on the output shape of the RNN.
        Example:
            # 1
            model.add(LSTM(64, return_sequences=True))
            model.add(Attention())
            # next add a Dense layer (for classification/regression) or whatever...
            # 2
            hidden = LSTM(64, return_sequences=True)(words)
            sentence = Attention()(hidden)
            # next add a Dense layer (for classification/regression) or whatever...
        """
        self.supports_masking = True
        self.init = initializers.get('glorot_uniform')

        self.W_regularizer = regularizers.get(W_regularizer)
        self.b_regularizer = regularizers.get(b_regularizer)

        self.W_constraint = constraints.get(W_constraint)
        self.b_constraint = constraints.get(b_constraint)

        self.bias = bias
        self.step_dim = step_dim
        self.features_dim = 0

        super(attention, self).__init__(**kwargs)

    def build(self, input_shape):
        assert len(input_shape) == 3

        self.W = self.add_weight(name='{}_W'.format(self.name),
                                 shape=(input_shape[-1],),
                                 initializer=self.init,
                                 regularizer=self.W_regularizer,
                                 constraint=self.W_constraint)
        self.features_dim = input_shape[-1]

        if self.bias:
            self.b = self.add_weight(name='{}_b'.format(self.name),
                                     shape=(input_shape[1],),
                                     initializer='zero',
                                     regularizer=self.b_regularizer,
                                     constraint=self.b_constraint)
        else:
            self.b = None

        self.built = True

    def compute_mask(self, input, input_mask=None):
        # do not pass the mask to the next layers
        return None

    def call(self, x, mask=None):
        features_dim = self.features_dim
        step_dim = self.step_dim

        e = K.reshape(K.dot(K.reshape(x, (-1, features_dim)), K.reshape(self.W, (features_dim, 1))), (-1, step_dim))  # e = K.dot(x, self.W)
        if self.bias:
            e += self.b
        e = K.tanh(e)

        a = K.exp(e)
        # apply mask after the exp. will be re-normalized next
        if mask is not None:
            # cast the mask to floatX to avoid float64 upcasting in theano
            a *= K.cast(mask, K.floatx())
        # in some cases especially in the early stages of training the sum may be almost zero
        # and this results in NaN's. A workaround is to add a very small positive number ε to the sum.
        a /= K.cast(K.sum(a, axis=1, keepdims=True) + K.epsilon(), K.floatx())
        a = K.expand_dims(a)

        c = K.sum(a * x, axis=1)
        return c

    def compute_output_shape(self, input_shape):
        return input_shape[0], self.features_dim

In [3]:
# this function takes orginal values and predicted values, 
#and outputs floor accuracy, average final measure (consisting MAE(Ecoord, Ncoord) and Floor Accuracy), MAE,
#RMSE, ..% localization errors, 

def calculate_performances(pred,y_valx):
   
    y_pred= pd.DataFrame(pred, columns = ['ECoord', 'NCoord','FloorID'])
    y_pred['FloorID']= y_pred['FloorID'].round()
   
    #calculating floor accuracy (firstly round predicted floorID because it is a regression result, which can not be integer)
    count=0
    floor_dist=[0 for i in range(len(pred))]
    for i in range(0,len(y_pred)):
        if abs(y_pred.FloorID.loc[i])==abs(y_valx.FloorID.loc[i]):
            count=count+1
            floor_dist[i]=1
   
    floor_acc=count/len(pred)   
    #print('Floor accuracy', floor_acc)

    del y_pred['FloorID']
    del y_valx['FloorID']

    #calculating AFS and MAE, (((len(pred)-count)*4)/len(pred)) refers to the number of false decision for floor 
    #if floor prediction is incorrect, an error of 4 meters is incurred, 
    #considering that the distance between floors is assumed to be 4 meters.
    
    avg_final_score= mean_absolute_error(y_valx, y_pred)+ (((len(pred)-count)*4)/len(pred))
    maex=mean_absolute_error(y_valx, y_pred)
    
    
    predicted_values = np.array(y_valx)
    ground_truth_values = np.array(y_pred)
    tempp= np.linalg.norm(predicted_values- ground_truth_values, axis=1)
    
    # Calculate the average localization error
    average_le = np.mean(tempp)
    
    squared_differences = (predicted_values- ground_truth_values) ** 2
    
    # Calculate the mean of the squared differences
    mean_squared_error = np.mean(squared_differences)
    
    # Calculate the root of the mean squared error
    #calculting RMSE
    rmsex2 = np.sqrt(mean_squared_error)
    
    

    final_dist=[0 for i in range(len(pred))]
    for i in range(0,len(floor_dist)):
        final_dist[i]=(abs(y_pred['ECoord'].loc[i]-y_valx['ECoord'].loc[i])+abs(y_pred['NCoord'].loc[i]-y_valx['NCoord'].loc[i]))/2
        if floor_dist[i]==0:
            final_dist[i]=final_dist[i]+4
    
    final_dist.sort()
   
    #ldistx=final_dist
    
    perc_pos=math.floor((50/100)*(len(final_dist)))

    if perc_pos>math.floor((50/100)*(len(final_dist))):
        error50=final_dist[perc_pos] + (final_dist[perc_pos+1]-final_dist[perc_pos])*0.5
    else:
        error50=final_dist[perc_pos]


    perc_pos=math.floor((75/100)*(len(final_dist)))

    if perc_pos>math.floor((75/100)*(len(final_dist))):
        error75=final_dist[perc_pos] + (final_dist[perc_pos+1]-final_dist[perc_pos])*0.5
    else:
        error75=final_dist[perc_pos]
        
    
    perc_pos=math.floor((95/100)*(len(final_dist)))

    if perc_pos>math.floor((95/100)*(len(final_dist))):
        error95=final_dist[perc_pos] + (final_dist[perc_pos+1]-final_dist[perc_pos])*0.5
    else:
        error95=final_dist[perc_pos]
    
    
    for i in range(0,len(final_dist)):
        final_dist[i]=pow(final_dist[i],2)

    from math import sqrt
    rmsex=sqrt(np.mean(final_dist)/2)
        

    return floor_acc, avg_final_score, maex, rmsex, rmsex2, error50, error75, error95, average_le

In [4]:
#create functions for different deep learning models

#inputs: inputlength, number of output for outputlayer(Ecoord, Ncoord, Floor id)
########################################################################################################
#Deep Neural Network (MLP)
def model_dnnreg(length,output_num):
    clf = Sequential()
    clf.add(Dense(256, input_dim =length, activation = 'relu'))
    clf.add(Dense(128, activation = 'relu'))
    clf.add(Dense(output_num, activation = 'linear'))
    clf.compile(loss = 'mean_absolute_error', optimizer = 'adam', metrics = ['mean_absolute_error'])
    return clf

########################################################################################################
#Convolutional Neural Network with two convolutional layer and one pooling
def model_CNNreg(length,output_num):

    inputs = Input(shape=(length,1))
    conv = Conv1D(filters=32, kernel_size=3, activation='relu')(inputs)
    conv = Conv1D(filters=32, kernel_size=3, activation='relu')(conv)
    pool = MaxPooling1D()(conv)
    flat=Flatten()(pool)
  
    dense = Dense(256, activation='relu')(flat)
    dense = Dense(128, activation='relu')(dense)
    
    outputs = Dense(output_num, activation='linear')(dense)
    model = Model(inputs=[inputs], outputs=outputs)
    # compile
    model.compile(loss='mean_absolute_error', optimizer='adam', metrics = ['mean_absolute_error'])
    print(model.summary())
    return model


########################################################################################################
#LSTM Network with one pooling layer
def model_LSTMreg(length,output_num):

    inputs = Input(shape=(length,1))
    lstm_layer =  LSTM(32, return_sequences=True)(inputs)
    pool = MaxPooling1D()(lstm_layer)
    flat=Flatten()(pool)
    dense = Dense(256, activation='relu')(flat)
    dense = Dense(128, activation='relu')(dense)
    outputs = Dense(output_num, activation='linear')(dense)
    model = Model(inputs=[inputs], outputs=outputs)
    # compile
    model.compile(loss='mean_absolute_error', optimizer='adam', metrics = ['mean_absolute_error'])
    print(model.summary())
    return model

########################################################################################################

#Multi-channel CNN network with attention layer (MC-ACNNR)
#inputs: inputlengthforchannel1, inputlengthforchannel2, number of output for outputlayer(Ecoord, Ncoord, Floor id)
def model_CNN_Multichannel_att(length1,length2,output_num,maxx):
  # channel 1
    inputs1 = Input(shape=(length1,1))
    inputs2 = Input(shape=(length2,1))

    conv1 = Conv1D(filters=32, kernel_size=3, padding='same', activation='relu')(inputs1)
    conv1 = Conv1D(filters=32, kernel_size=3, padding='same', activation='relu')(conv1)

    att1=attention(maxx)(conv1)
   
    conv2 = Conv1D(filters=32, kernel_size=3, padding='same', activation='relu')(inputs2)
    conv2 = Conv1D(filters=32, kernel_size=3, padding='same', activation='relu')(conv2)

    att2=attention(maxx)(conv2)
    
    merged=concatenate([conv1,conv2])
    conv = Conv1D(filters=32, kernel_size=3, padding='same',  activation='relu')(merged)
    conv = Conv1D(filters=32, kernel_size=3, padding='same',  activation='relu')(conv)

    pool = MaxPooling1D()(conv)
    flat=Flatten()(pool)
    
    flat=concatenate([att1,flat,att2])
    
    dense = Dense(256, activation='relu')(flat)
    dense = Dense(128, activation='relu')(dense)
    outputs = Dense(output_num, activation='linear')(dense)
    model = Model(inputs=[inputs1, inputs2], outputs=outputs)
    # compile
    model.compile(loss='mean_absolute_error', optimizer='adam', metrics = ['mean_absolute_error'])   
    # summarize
    print(model.summary())
    return model

In [15]:
#SOD_SYL
#DNNR

#K.clear_session()
tf.keras.backend.clear_session()

learning_rate=0.001
batch_size=100
epochs=250

early_stop = EarlyStopping(monitor = 'val_loss', min_delta = 0.0001, 
                       patience = 8, mode = 'min', verbose = 1,
                       restore_best_weights = True)
reduce_lr = ReduceLROnPlateau(monitor = 'val_loss', factor = 0.6, 
                          patience = 5, min_delta = 0.0001, 
                          mode = 'min', verbose = 1)
    
columns= {'iter':[],
        'Model':[], 
        'ACC':[], 
        'AFS':[],
        'MAE':[], 
        'RMSE':[],
        'RMSE2':[],
        '50th%error':[],
        '75th%error':[],
        '95th%error':[],
        'avg_le':[]
       } 


df_comp= pd.DataFrame(columns)

filepath='C:/Users/user/Desktop/arzu_kodlar/indoor_diger/checkpoints/'


df=pd.read_csv('data/SYL_preprocessed/SOD_SYL_24_merged_normalized_dropped.csv')
df = df.astype(float)

df_train = df[df['datatype'] == 1]
df_train=df_train.reset_index(drop=True)
df_test = df[df['datatype'] == 2]
df_test=df_test.reset_index(drop=True)

y_train= df_train[['ECoord', 'NCoord','FloorID']]
y_val=df_test[['ECoord', 'NCoord','FloorID']]

x_train=df_train.drop(['datatype','ECoord', 'NCoord','FloorID'], axis=1)
x_val=df_test.drop(['datatype','ECoord', 'NCoord','FloorID'], axis=1)

x_trainx = tf.convert_to_tensor(x_train)
y_trainx = tf.convert_to_tensor(y_train)

x_valx = tf.convert_to_tensor(x_val)
y_valx = tf.convert_to_tensor(y_val)
    

model_name='DNNR_24GHz'

for iter in range(0,3):
    
    y_val=df_test[['ECoord', 'NCoord','FloorID']]
    y_valx = tf.convert_to_tensor(y_val)


    checkpoint_filepath = filepath +model_name+ str(iter)+ '2x24.hdf5'

    model_checkpoint_callback = ModelCheckpoint(
    filepath=checkpoint_filepath,
    save_weights_only=True,
    monitor='val_mean_absolute_error',
    mode='min',
    save_best_only=True)
    callbacks = [model_checkpoint_callback,early_stop,reduce_lr]
    
    
    deep_model=model_dnnreg(x_train.shape[1], y_train.shape[1])
    
    deep_model.fit(x_trainx, y_trainx, epochs=epochs, batch_size=batch_size, callbacks =callbacks, validation_data=(x_valx, y_valx))
    
    deep_model.load_weights(checkpoint_filepath)
    
    pred= deep_model.predict(x_val)

    floor_acc, avg_final_score, maex, rmsex, rmsex2, error50, error75, error95, average_le=calculate_performances(pred, y_val)   

    df_comp.loc[len(df_comp.index)] = [iter, model_name, floor_acc, avg_final_score, maex, rmsex, rmsex2, error50, error75, error95, average_le]  
    
    display(df_comp)


df_comp.to_csv("SOD_SYL_24_DNNR.csv",index=False)

Epoch 1/250
7/7 [==============================] - 1s 45ms/step - loss: 21.4033 - mean_absolute_error: 21.4033 - val_loss: 21.0204 - val_mean_absolute_error: 21.0204 - lr: 0.0010
Epoch 2/250
7/7 [==============================] - 0s 14ms/step - loss: 20.7187 - mean_absolute_error: 20.7187 - val_loss: 19.9874 - val_mean_absolute_error: 19.9874 - lr: 0.0010
Epoch 3/250
7/7 [==============================] - 0s 13ms/step - loss: 19.5748 - mean_absolute_error: 19.5748 - val_loss: 18.3100 - val_mean_absolute_error: 18.3100 - lr: 0.0010
Epoch 4/250
7/7 [==============================] - 0s 14ms/step - loss: 17.8093 - mean_absolute_error: 17.8093 - val_loss: 16.5085 - val_mean_absolute_error: 16.5085 - lr: 0.0010
Epoch 5/250
7/7 [==============================] - 0s 13ms/step - loss: 16.0439 - mean_absolute_error: 16.0439 - val_loss: 14.7647 - val_mean_absolute_error: 14.7647 - lr: 0.0010
Epoch 6/250
7/7 [==============================] - 0s 15ms/step - loss: 14.2033 - mean_absolute_error: 14

7/7 [==============================] - 0s 9ms/step - loss: 2.2253 - mean_absolute_error: 2.2253 - val_loss: 2.8839 - val_mean_absolute_error: 2.8839 - lr: 6.0000e-04
Epoch 47/250
7/7 [==============================] - 0s 10ms/step - loss: 2.2214 - mean_absolute_error: 2.2214 - val_loss: 2.8927 - val_mean_absolute_error: 2.8927 - lr: 6.0000e-04
Epoch 48/250
7/7 [==============================] - 0s 11ms/step - loss: 2.2097 - mean_absolute_error: 2.2097 - val_loss: 2.8732 - val_mean_absolute_error: 2.8732 - lr: 6.0000e-04
Epoch 49/250
7/7 [==============================] - 0s 10ms/step - loss: 2.1980 - mean_absolute_error: 2.1980 - val_loss: 2.8899 - val_mean_absolute_error: 2.8899 - lr: 6.0000e-04
Epoch 50/250
1/7 [===>..........................] - ETA: 0s - loss: 2.1795 - mean_absolute_error: 2.1795
Epoch 50: ReduceLROnPlateau reducing learning rate to 0.0003600000170990825.
7/7 [==============================] - 0s 11ms/step - loss: 2.1926 - mean_absolute_error: 2.1926 - val_loss: 2.8

,iter,Model,ACC,AFS,MAE,RMSE,RMSE2,50th%error,75th%error,95th%error,avg_le
0,0,DNNR_24GHz,1.0,4.122421,4.122421,4.033235,7.09753,3.539617,4.999208,8.284059,6.840364


Epoch 1/250
7/7 [==============================] - 1s 44ms/step - loss: 21.4028 - mean_absolute_error: 21.4028 - val_loss: 21.0382 - val_mean_absolute_error: 21.0382 - lr: 0.0010
Epoch 2/250
7/7 [==============================] - 0s 12ms/step - loss: 20.7267 - mean_absolute_error: 20.7267 - val_loss: 20.0015 - val_mean_absolute_error: 20.0015 - lr: 0.0010
Epoch 3/250
7/7 [==============================] - 0s 14ms/step - loss: 19.5335 - mean_absolute_error: 19.5335 - val_loss: 18.4268 - val_mean_absolute_error: 18.4268 - lr: 0.0010
Epoch 4/250
7/7 [==============================] - 0s 13ms/step - loss: 18.0315 - mean_absolute_error: 18.0315 - val_loss: 17.0593 - val_mean_absolute_error: 17.0593 - lr: 0.0010
Epoch 5/250
7/7 [==============================] - 0s 13ms/step - loss: 16.4517 - mean_absolute_error: 16.4517 - val_loss: 14.9817 - val_mean_absolute_error: 14.9817 - lr: 0.0010
Epoch 6/250
7/7 [==============================] - 0s 13ms/step - loss: 14.3362 - mean_absolute_error: 14

7/7 [==============================] - 0s 13ms/step - loss: 2.1970 - mean_absolute_error: 2.1970 - val_loss: 2.8316 - val_mean_absolute_error: 2.8316 - lr: 0.0010
Epoch 48/250
7/7 [==============================] - 0s 9ms/step - loss: 2.1754 - mean_absolute_error: 2.1754 - val_loss: 2.8569 - val_mean_absolute_error: 2.8569 - lr: 0.0010
Epoch 49/250
7/7 [==============================] - 0s 10ms/step - loss: 2.1538 - mean_absolute_error: 2.1538 - val_loss: 2.8735 - val_mean_absolute_error: 2.8735 - lr: 0.0010
Epoch 50/250
7/7 [==============================] - 0s 13ms/step - loss: 2.1494 - mean_absolute_error: 2.1494 - val_loss: 2.7716 - val_mean_absolute_error: 2.7716 - lr: 0.0010
Epoch 51/250
7/7 [==============================] - 0s 10ms/step - loss: 2.1496 - mean_absolute_error: 2.1496 - val_loss: 2.8442 - val_mean_absolute_error: 2.8442 - lr: 0.0010
Epoch 52/250
7/7 [==============================] - 0s 9ms/step - loss: 2.1274 - mean_absolute_error: 2.1274 - val_loss: 2.7952 - val_

,iter,Model,ACC,AFS,MAE,RMSE,RMSE2,50th%error,75th%error,95th%error,avg_le
0,0,DNNR_24GHz,1.0,4.122421,4.122421,4.033235,7.097530,3.539617,4.999208,8.284059,6.840364
1,1,DNNR_24GHz,1.0,3.922138,3.922138,3.872349,6.869014,3.310868,4.458023,7.697840,6.575005


Epoch 1/250
7/7 [==============================] - 1s 45ms/step - loss: 21.3650 - mean_absolute_error: 21.3650 - val_loss: 20.9068 - val_mean_absolute_error: 20.9068 - lr: 0.0010
Epoch 2/250
7/7 [==============================] - 0s 13ms/step - loss: 20.5737 - mean_absolute_error: 20.5737 - val_loss: 19.7494 - val_mean_absolute_error: 19.7494 - lr: 0.0010
Epoch 3/250
7/7 [==============================] - 0s 13ms/step - loss: 19.3075 - mean_absolute_error: 19.3075 - val_loss: 18.3827 - val_mean_absolute_error: 18.3827 - lr: 0.0010
Epoch 4/250
7/7 [==============================] - 0s 12ms/step - loss: 17.9954 - mean_absolute_error: 17.9954 - val_loss: 16.9998 - val_mean_absolute_error: 16.9998 - lr: 0.0010
Epoch 5/250
7/7 [==============================] - 0s 14ms/step - loss: 16.4271 - mean_absolute_error: 16.4271 - val_loss: 15.0317 - val_mean_absolute_error: 15.0317 - lr: 0.0010
Epoch 6/250
7/7 [==============================] - 0s 13ms/step - loss: 14.4273 - mean_absolute_error: 14

7/7 [==============================] - 0s 9ms/step - loss: 2.1462 - mean_absolute_error: 2.1462 - val_loss: 2.8615 - val_mean_absolute_error: 2.8615 - lr: 0.0010
Epoch 48/250
7/7 [==============================] - 0s 14ms/step - loss: 2.1230 - mean_absolute_error: 2.1230 - val_loss: 2.8416 - val_mean_absolute_error: 2.8416 - lr: 0.0010
Epoch 49/250
7/7 [==============================] - 0s 13ms/step - loss: 2.1173 - mean_absolute_error: 2.1173 - val_loss: 2.8351 - val_mean_absolute_error: 2.8351 - lr: 0.0010
Epoch 50/250
7/7 [==============================] - 0s 13ms/step - loss: 2.1111 - mean_absolute_error: 2.1111 - val_loss: 2.8179 - val_mean_absolute_error: 2.8179 - lr: 0.0010
Epoch 51/250
7/7 [==============================] - 0s 10ms/step - loss: 2.0846 - mean_absolute_error: 2.0846 - val_loss: 2.8534 - val_mean_absolute_error: 2.8534 - lr: 0.0010
Epoch 52/250
7/7 [==============================] - 0s 13ms/step - loss: 2.0776 - mean_absolute_error: 2.0776 - val_loss: 2.7684 - val

,iter,Model,ACC,AFS,MAE,RMSE,RMSE2,50th%error,75th%error,95th%error,avg_le
0,0,DNNR_24GHz,1.0,4.122421,4.122421,4.033235,7.097530,3.539617,4.999208,8.284059,6.840364
1,1,DNNR_24GHz,1.0,3.922138,3.922138,3.872349,6.869014,3.310868,4.458023,7.697840,6.575005
2,2,DNNR_24GHz,1.0,4.001253,4.001253,3.934640,6.972718,3.347883,4.981908,7.995031,6.683080


In [17]:
#SOD_SYL
#MC_ACNNR

learning_rate=0.001
batch_size=64
epochs=1000

early_stop = EarlyStopping(monitor = 'val_loss', min_delta = 0.0001, 
                       patience = 8, mode = 'min', verbose = 1,
                       restore_best_weights = True)
reduce_lr = ReduceLROnPlateau(monitor = 'val_loss', factor = 0.6, 
                          patience = 4, min_delta = 0.0001, 
                          mode = 'min', verbose = 1)
    
columns= {'iter':[],
        'Model':[], 
        'ACC':[], 
        'AFS':[],
        'MAE':[], 
        'RMSE':[],
        'RMSE2':[],
        '50th%error':[],
        '75th%error':[],
        '95th%error':[],
        'avg_le':[]
       } 

df_comp= pd.DataFrame(columns)

filepath='C:/Users/user/Desktop/arzu_kodlar/indoor_diger/checkpoints/'

df=pd.read_csv('data/SYL_preprocessed/SOD_SYL_dual_merged_normalized_nondropped_testsamples.csv')
df = df.astype(float)

df_train = df[df['datatype'] == 1]
df_train=df_train.reset_index(drop=True)
df_test = df[df['datatype'] == 2]
df_test=df_test.reset_index(drop=True)

y_train= df_train[['ECoord', 'NCoord','FloorID']]
y_val=df_test[['ECoord', 'NCoord','FloorID']]

x_train=df_train.drop(['datatype','ECoord', 'NCoord','FloorID'], axis=1)
x_val=df_test.drop(['datatype','ECoord', 'NCoord','FloorID'], axis=1)

x_trainx = tf.convert_to_tensor(x_train)
y_trainx = tf.convert_to_tensor(y_train)

x_valx = tf.convert_to_tensor(x_val)
y_valx = tf.convert_to_tensor(y_val)
    

#each GHz area consits of 23 macs
maxlen=23

#if the input length is not equal to each other, you need to do zero padding for making equal them
#x_trainx=np.pad(x_train, ((0, 0), (0, 16)))
#x_valx=np.pad(x_val, ((0, 0), (0, 16)))


model_name='MC-ACNNR_dual'

for iter in range(0,3):
    
    y_val=df_test[['ECoord', 'NCoord','FloorID']]
    y_valx = tf.convert_to_tensor(y_val)

    checkpoint_filepath = filepath +model_name+ str(iter)+ '2.hdf5'

    model_checkpoint_callback = ModelCheckpoint(
    filepath=checkpoint_filepath,
    save_weights_only=True,
    monitor='val_mean_absolute_error',
    mode='min',
    save_best_only=True)
    
    #callbacks = [model_checkpoint_callback,early_stop,reduce_lr]
    callbacks = [model_checkpoint_callback]

    deep_model=model_CNN_Multichannel_att(maxlen,maxlen,y_val.shape[1],maxlen)

    deep_model.fit([x_trainx[:,0:maxlen],x_trainx[:,maxlen:maxlen*2]], y_trainx, epochs=epochs, batch_size=batch_size, callbacks=callbacks, validation_data=([x_valx[:,0:maxlen],x_valx[:,maxlen:maxlen*2]], y_valx))

    deep_model.load_weights(checkpoint_filepath)
    
    pred=deep_model.predict([x_valx[:,0:maxlen],x_valx[:,maxlen:maxlen*2]])
 
    floor_acc, avg_final_score, maex, rmsex, rmsex2, error50, error75, error95, average_le=calculate_performances(pred, y_val)   

    df_comp.loc[len(df_comp.index)] = [iter, model_name, floor_acc, avg_final_score, maex, rmsex, rmsex2, error50, error75, error95, average_le]  
    
    display(df_comp)

df_comp.to_csv("SOD_SYL_MC-ACNNR_nondrop.csv",index=False)

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_3 (InputLayer)           [(None, 23, 1)]      0           []                               
                                                                                                  
 input_4 (InputLayer)           [(None, 23, 1)]      0           []                               
                                                                                                  
 conv1d_6 (Conv1D)              (None, 23, 32)       128         ['input_3[0][0]']                
                                                                                                  
 conv1d_8 (Conv1D)              (None, 23, 32)       128         ['input_4[0][0]']                
                                                                                            

Epoch 25/1000
11/11 [==============================] - 0s 36ms/step - loss: 2.1795 - mean_absolute_error: 2.1795 - val_loss: 2.5206 - val_mean_absolute_error: 2.5206
Epoch 26/1000
11/11 [==============================] - 0s 25ms/step - loss: 2.4864 - mean_absolute_error: 2.4864 - val_loss: 3.2070 - val_mean_absolute_error: 3.2070
Epoch 27/1000
11/11 [==============================] - 0s 20ms/step - loss: 2.2031 - mean_absolute_error: 2.2031 - val_loss: 2.5640 - val_mean_absolute_error: 2.5640
Epoch 28/1000
11/11 [==============================] - 0s 23ms/step - loss: 2.1763 - mean_absolute_error: 2.1763 - val_loss: 2.5709 - val_mean_absolute_error: 2.5709
Epoch 29/1000
11/11 [==============================] - 0s 31ms/step - loss: 2.0992 - mean_absolute_error: 2.0992 - val_loss: 2.6448 - val_mean_absolute_error: 2.6448
Epoch 30/1000
11/11 [==============================] - 0s 27ms/step - loss: 2.0485 - mean_absolute_error: 2.0485 - val_loss: 2.3776 - val_mean_absolute_error: 2.3776
Epoc

11/11 [==============================] - 0s 16ms/step - loss: 1.4202 - mean_absolute_error: 1.4202 - val_loss: 2.0322 - val_mean_absolute_error: 2.0322
Epoch 75/1000
11/11 [==============================] - 0s 16ms/step - loss: 1.3599 - mean_absolute_error: 1.3599 - val_loss: 2.1701 - val_mean_absolute_error: 2.1701
Epoch 76/1000
11/11 [==============================] - 0s 15ms/step - loss: 1.3544 - mean_absolute_error: 1.3544 - val_loss: 2.2785 - val_mean_absolute_error: 2.2785
Epoch 77/1000
11/11 [==============================] - 0s 17ms/step - loss: 1.4127 - mean_absolute_error: 1.4127 - val_loss: 2.0942 - val_mean_absolute_error: 2.0942
Epoch 78/1000
11/11 [==============================] - 0s 21ms/step - loss: 1.3838 - mean_absolute_error: 1.3838 - val_loss: 1.9925 - val_mean_absolute_error: 1.9925
Epoch 79/1000
11/11 [==============================] - 0s 15ms/step - loss: 1.3713 - mean_absolute_error: 1.3713 - val_loss: 2.1663 - val_mean_absolute_error: 2.1663
Epoch 80/1000
11/1

11/11 [==============================] - 0s 16ms/step - loss: 0.9056 - mean_absolute_error: 0.9056 - val_loss: 1.7916 - val_mean_absolute_error: 1.7916
Epoch 124/1000
11/11 [==============================] - 0s 18ms/step - loss: 0.9079 - mean_absolute_error: 0.9079 - val_loss: 1.7905 - val_mean_absolute_error: 1.7905
Epoch 125/1000
11/11 [==============================] - 0s 23ms/step - loss: 1.1119 - mean_absolute_error: 1.1119 - val_loss: 1.7103 - val_mean_absolute_error: 1.7103
Epoch 126/1000
11/11 [==============================] - 0s 15ms/step - loss: 1.0347 - mean_absolute_error: 1.0347 - val_loss: 2.0365 - val_mean_absolute_error: 2.0365
Epoch 127/1000
11/11 [==============================] - 0s 16ms/step - loss: 0.9285 - mean_absolute_error: 0.9285 - val_loss: 1.7974 - val_mean_absolute_error: 1.7974
Epoch 128/1000
11/11 [==============================] - 0s 15ms/step - loss: 0.9175 - mean_absolute_error: 0.9175 - val_loss: 2.0120 - val_mean_absolute_error: 2.0120
Epoch 129/100

11/11 [==============================] - 0s 15ms/step - loss: 0.7489 - mean_absolute_error: 0.7489 - val_loss: 1.8612 - val_mean_absolute_error: 1.8612
Epoch 173/1000
11/11 [==============================] - 0s 19ms/step - loss: 0.6618 - mean_absolute_error: 0.6618 - val_loss: 1.7252 - val_mean_absolute_error: 1.7252
Epoch 174/1000
11/11 [==============================] - 0s 16ms/step - loss: 0.7253 - mean_absolute_error: 0.7253 - val_loss: 1.7974 - val_mean_absolute_error: 1.7974
Epoch 175/1000
11/11 [==============================] - 0s 16ms/step - loss: 0.7487 - mean_absolute_error: 0.7487 - val_loss: 1.8161 - val_mean_absolute_error: 1.8161
Epoch 176/1000
11/11 [==============================] - 0s 17ms/step - loss: 0.8011 - mean_absolute_error: 0.8011 - val_loss: 1.9043 - val_mean_absolute_error: 1.9043
Epoch 177/1000
11/11 [==============================] - 0s 15ms/step - loss: 0.7032 - mean_absolute_error: 0.7032 - val_loss: 1.8177 - val_mean_absolute_error: 1.8177
Epoch 178/100

...

11/11 [==============================] - 0s 33ms/step - loss: 0.2469 - mean_absolute_error: 0.2469 - val_loss: 1.8750 - val_mean_absolute_error: 1.8750
Epoch 810/1000
11/11 [==============================] - 0s 32ms/step - loss: 0.2601 - mean_absolute_error: 0.2601 - val_loss: 1.8386 - val_mean_absolute_error: 1.8386
Epoch 811/1000
11/11 [==============================] - 0s 32ms/step - loss: 0.2426 - mean_absolute_error: 0.2426 - val_loss: 1.8734 - val_mean_absolute_error: 1.8734
Epoch 812/1000
11/11 [==============================] - 0s 27ms/step - loss: 0.2871 - mean_absolute_error: 0.2871 - val_loss: 1.8536 - val_mean_absolute_error: 1.8536
Epoch 813/1000
11/11 [==============================] - 0s 17ms/step - loss: 0.3176 - mean_absolute_error: 0.3176 - val_loss: 1.8186 - val_mean_absolute_error: 1.8186
Epoch 814/1000
11/11 [==============================] - 0s 17ms/step - loss: 0.3673 - mean_absolute_error: 0.3673 - val_loss: 1.9347 - val_mean_absolute_error: 1.9347
Epoch 815/100

11/11 [==============================] - 0s 22ms/step - loss: 0.2954 - mean_absolute_error: 0.2954 - val_loss: 1.8972 - val_mean_absolute_error: 1.8972
Epoch 859/1000
11/11 [==============================] - 0s 19ms/step - loss: 0.2539 - mean_absolute_error: 0.2539 - val_loss: 1.9341 - val_mean_absolute_error: 1.9341
Epoch 860/1000
11/11 [==============================] - 0s 16ms/step - loss: 0.3469 - mean_absolute_error: 0.3469 - val_loss: 1.8170 - val_mean_absolute_error: 1.8170
Epoch 861/1000
11/11 [==============================] - 0s 16ms/step - loss: 0.2894 - mean_absolute_error: 0.2894 - val_loss: 1.8610 - val_mean_absolute_error: 1.8610
Epoch 862/1000
11/11 [==============================] - 0s 17ms/step - loss: 0.2684 - mean_absolute_error: 0.2684 - val_loss: 1.8622 - val_mean_absolute_error: 1.8622
Epoch 863/1000
11/11 [==============================] - 0s 20ms/step - loss: 0.2573 - mean_absolute_error: 0.2573 - val_loss: 1.8524 - val_mean_absolute_error: 1.8524
Epoch 864/100

11/11 [==============================] - 0s 19ms/step - loss: 0.2601 - mean_absolute_error: 0.2601 - val_loss: 1.8318 - val_mean_absolute_error: 1.8318
Epoch 908/1000
11/11 [==============================] - 0s 24ms/step - loss: 0.2290 - mean_absolute_error: 0.2290 - val_loss: 1.8560 - val_mean_absolute_error: 1.8560
Epoch 909/1000
11/11 [==============================] - 0s 20ms/step - loss: 0.2121 - mean_absolute_error: 0.2121 - val_loss: 1.8625 - val_mean_absolute_error: 1.8625
Epoch 910/1000
11/11 [==============================] - 0s 21ms/step - loss: 0.2538 - mean_absolute_error: 0.2538 - val_loss: 1.8601 - val_mean_absolute_error: 1.8601
Epoch 911/1000
11/11 [==============================] - 0s 31ms/step - loss: 0.2667 - mean_absolute_error: 0.2667 - val_loss: 1.8284 - val_mean_absolute_error: 1.8284
Epoch 912/1000
11/11 [==============================] - 0s 17ms/step - loss: 0.2570 - mean_absolute_error: 0.2570 - val_loss: 1.8484 - val_mean_absolute_error: 1.8484
Epoch 913/100

11/11 [==============================] - 0s 17ms/step - loss: 0.3256 - mean_absolute_error: 0.3256 - val_loss: 1.8904 - val_mean_absolute_error: 1.8904
Epoch 957/1000
11/11 [==============================] - 0s 20ms/step - loss: 0.2623 - mean_absolute_error: 0.2623 - val_loss: 1.8110 - val_mean_absolute_error: 1.8110
Epoch 958/1000
11/11 [==============================] - 0s 17ms/step - loss: 0.2571 - mean_absolute_error: 0.2571 - val_loss: 1.8356 - val_mean_absolute_error: 1.8356
Epoch 959/1000
11/11 [==============================] - 0s 16ms/step - loss: 0.2676 - mean_absolute_error: 0.2676 - val_loss: 1.9081 - val_mean_absolute_error: 1.9081
Epoch 960/1000
11/11 [==============================] - 0s 17ms/step - loss: 0.2227 - mean_absolute_error: 0.2227 - val_loss: 1.8580 - val_mean_absolute_error: 1.8580
Epoch 961/1000
11/11 [==============================] - 0s 17ms/step - loss: 0.2454 - mean_absolute_error: 0.2454 - val_loss: 1.9113 - val_mean_absolute_error: 1.9113
Epoch 962/100

,iter,Model,ACC,AFS,MAE,RMSE,RMSE2,50th%error,75th%error,95th%error,avg_le
0,0,MC-ACNNR_dual,1.0,2.499237,2.499237,2.539453,4.460683,2.097947,3.446978,4.812274,4.094710
1,1,MC-ACNNR_dual,1.0,2.526197,2.526197,2.567692,4.501695,1.998325,3.113951,5.415851,4.159268
2,2,MC-ACNNR_dual,1.0,2.610890,2.610890,2.664917,4.747486,2.301337,3.189365,5.103476,4.355452
